In [2]:
!pip install -q pyspark==3.5.1

In [3]:
import os
import shutil
import numpy as np
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.regression import LinearRegression

from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator,
    BinaryClassificationEvaluator,
    RegressionEvaluator
)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [4]:
spark = (
    SparkSession.builder
    .appName("AI_ML_PowerBI_End_to_End")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 3.5.1


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Assignment_DIR = "/content/drive/My Drive/Lab_BigData/AI_ML_PowerBI_Assignment"
os.makedirs(Assignment_DIR, exist_ok=True)
print("Assignment directory:", Assignment_DIR)

Mounted at /content/drive
Assignment directory: /content/drive/My Drive/Lab_BigData/AI_ML_PowerBI_Assignment


In [8]:
import os
print(os.listdir("/content"))

['.config', 'drive', '01_raw_customer_data (2).csv', 'sample_data']


In [9]:
raw_file = "/content/01_raw_customer_data (2).csv"

df = spark.read.csv(raw_file, header=True, inferSchema=True)
df.printSchema()
df.show(10, truncate=False)

root
 |-- customer_id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- income: double (nullable = true)
 |-- tenure_months: integer (nullable = true)
 |-- monthly_spend: double (nullable = true)
 |-- usage_hours: double (nullable = true)
 |-- support_calls: integer (nullable = true)
 |-- satisfaction: double (nullable = true)
 |-- churn: integer (nullable = true)
 |-- next_month_spend: double (nullable = true)

+-----------+---+--------+-------------+-------------+-----------+-------------+------------+-----+----------------+
|customer_id|age|income  |tenure_months|monthly_spend|usage_hours|support_calls|satisfaction|churn|next_month_spend|
+-----------+---+--------+-------------+-------------+-----------+-------------+------------+-----+----------------+
|100001     |56 |44657.04|105          |844.11       |40.87      |0            |6.58        |0    |2240.59         |
|100002     |69 |68556.58|81           |3266.18      |12.28      |4            |5.66        |0    

In [10]:
print("Number of rows:", df.count())
print("Number of columns:", len(df.columns))
df.describe().show()

missing = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])
missing.show()

Number of rows: 10000
Number of columns: 10
+-------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+-------------------+------------------+
|summary|       customer_id|               age|            income|     tenure_months|    monthly_spend|       usage_hours|     support_calls|      satisfaction|              churn|  next_month_spend|
+-------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+-------------------+------------------+
|  count|             10000|             10000|             10000|             10000|            10000|             10000|             10000|             10000|              10000|             10000|
|   mean|          105000.5|           43.5394|      60089.296922|           60.7008| 3007.82414399998|34.992974000000125|            2.0097| 6.489019999999

In [11]:
clean_df = df.dropDuplicates(["customer_id"]).dropna()
print("Rows after cleaning:", clean_df.count())

Rows after cleaning: 10000


In [12]:
import os

Assignment_DIR = "/content/AI_ML_PowerBI_Assignment"
os.makedirs(Assignment_DIR, exist_ok=True)

clean_pdf = clean_df.toPandas()
clean_file = os.path.join(Assignment_DIR, "02_clean_customer_data.csv")
clean_pdf.to_csv(clean_file, index=False)
print("Saved:", clean_file)

Saved: /content/AI_ML_PowerBI_Assignment/02_clean_customer_data.csv


In [13]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression

classification_features = [
    "age", "income", "tenure_months", "monthly_spend",
    "usage_hours", "support_calls", "satisfaction"
]

assembler_cls = VectorAssembler(inputCols=classification_features, outputCol="features_raw")
scaler_cls = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=True)

lr = LogisticRegression(
    featuresCol="features", labelCol="churn",
    predictionCol="prediction", probabilityCol="probability",
    rawPredictionCol="rawPrediction", maxIter=100
)

classification_pipeline = Pipeline(stages=[assembler_cls, scaler_cls, lr])

In [14]:
train_cls, test_cls = clean_df.randomSplit([0.8, 0.2], seed=42)
print("Training records:", train_cls.count())
print("Testing records:", test_cls.count())

Training records: 8079
Testing records: 1921


In [15]:
classification_model = classification_pipeline.fit(train_cls)
print("Classification model trained successfully.")

classification_predictions = classification_model.transform(test_cls)
classification_predictions.select("customer_id", "churn", "prediction", "probability").show(10, truncate=False)

Classification model trained successfully.
+-----------+-----+----------+-----------------------------------------+
|customer_id|churn|prediction|probability                              |
+-----------+-----+----------+-----------------------------------------+
|100003     |0    |0.0       |[0.9951264944898468,0.004873505510153198]|
|100007     |0    |0.0       |[0.9991091205053776,8.908794946224097E-4]|
|100009     |0    |0.0       |[0.9812735146678018,0.01872648533219823] |
|100014     |0    |0.0       |[0.9858707210356685,0.014129278964331471]|
|100020     |0    |0.0       |[0.9977227172077249,0.002277282792275126]|
|100024     |0    |0.0       |[0.9988225647908803,0.001177435209119726]|
|100030     |0    |0.0       |[0.994512960606698,0.005487039393301973] |
|100036     |0    |0.0       |[0.9915192986446711,0.008480701355328857]|
|100046     |0    |0.0       |[0.9962576168176661,0.003742383182333886]|
|100047     |0    |0.0       |[0.9731424826187329,0.02685751738126707] |
+-------

In [16]:
classification_model.save(os.path.join(Assignment_DIR, "classification_model"))
print("Classification model saved.")

Classification model saved.


In [17]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

accuracy_evaluator = MulticlassClassificationEvaluator(labelCol="churn", predictionCol="prediction", metricName="accuracy")
f1_evaluator = MulticlassClassificationEvaluator(labelCol="churn", predictionCol="prediction", metricName="f1")
precision_evaluator = MulticlassClassificationEvaluator(labelCol="churn", predictionCol="prediction", metricName="weightedPrecision")
recall_evaluator = MulticlassClassificationEvaluator(labelCol="churn", predictionCol="prediction", metricName="weightedRecall")
auc_evaluator = BinaryClassificationEvaluator(labelCol="churn", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

accuracy = accuracy_evaluator.evaluate(classification_predictions)
f1 = f1_evaluator.evaluate(classification_predictions)
precision = precision_evaluator.evaluate(classification_predictions)
recall = recall_evaluator.evaluate(classification_predictions)
auc = auc_evaluator.evaluate(classification_predictions)

print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))
print("ROC-AUC  :", round(auc, 4))

Accuracy : 0.9859
Precision: 0.9721
Recall   : 0.9859
F1 Score : 0.979
ROC-AUC  : 0.7527


In [18]:
import pandas as pd

confusion_matrix = (
    classification_predictions
    .groupBy("churn", "prediction")
    .count()
    .orderBy("churn", "prediction")
)
confusion_matrix.show()

cm_pdf = confusion_matrix.toPandas()
cm_pdf["Actual"] = cm_pdf["churn"].map({0: "No Churn", 1: "Churn"})
cm_pdf["Predicted"] = cm_pdf["prediction"].map({0: "No Churn", 1: "Churn"})
cm_pdf = cm_pdf[["Actual", "Predicted", "count"]]

cm_file = os.path.join(Assignment_DIR, "05_confusion_matrix.csv")
cm_pdf.to_csv(cm_file, index=False)
cm_pdf

+-----+----------+-----+
|churn|prediction|count|
+-----+----------+-----+
|    0|       0.0| 1894|
|    1|       0.0|   27|
+-----+----------+-----+



,Actual,Predicted,count
0,No Churn,No Churn,1894
1,Churn,No Churn,27


In [19]:
classification_metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "Value": [accuracy, precision, recall, f1, auc]
})
classification_metrics["Model"] = "Logistic Regression"

classification_metrics_file = os.path.join(Assignment_DIR, "04_classification_metrics.csv")
classification_metrics.to_csv(classification_metrics_file, index=False)
classification_metrics

,Metric,Value,Model
0,Accuracy,0.985945,Logistic Regression
1,Precision,0.972087,Logistic Regression
2,Recall,0.985945,Logistic Regression
3,F1 Score,0.978967,Logistic Regression
4,ROC-AUC,0.752708,Logistic Regression


In [20]:
cls_output = classification_predictions.select(
    "customer_id", "age", "income", "tenure_months", "monthly_spend",
    "usage_hours", "support_calls", "satisfaction", "churn", "prediction"
)
cls_pdf = cls_output.toPandas()
cls_pdf["Actual_Label"] = cls_pdf["churn"].map({0: "No Churn", 1: "Churn"})
cls_pdf["Predicted_Label"] = cls_pdf["prediction"].map({0: "No Churn", 1: "Churn"})

cls_file = os.path.join(Assignment_DIR, "03_classification_predictions.csv")
cls_pdf.to_csv(cls_file, index=False)
print("Saved:", cls_file)

Saved: /content/AI_ML_PowerBI_Assignment/03_classification_predictions.csv


In [21]:
from pyspark.ml.regression import LinearRegression

regression_features = [
    "age", "income", "tenure_months", "monthly_spend",
    "usage_hours", "support_calls", "satisfaction", "churn"
]

assembler_reg = VectorAssembler(inputCols=regression_features, outputCol="features")
linear_regression = LinearRegression(featuresCol="features", labelCol="next_month_spend", predictionCol="prediction")

regression_pipeline = Pipeline(stages=[assembler_reg, linear_regression])

In [22]:
train_reg, test_reg = clean_df.randomSplit([0.8, 0.2], seed=42)
print("Training:", train_reg.count())
print("Testing :", test_reg.count())

regression_model = regression_pipeline.fit(train_reg)
print("Regression model trained.")

regression_model.save(os.path.join(Assignment_DIR, "regression_model"))
print("Regression model saved.")

Training: 8079
Testing : 1921
Regression model trained.
Regression model saved.


In [23]:
from pyspark.ml.evaluation import RegressionEvaluator

regression_predictions = regression_model.transform(test_reg)
regression_predictions.select("customer_id", "next_month_spend", "prediction").show(10)

mae_evaluator = RegressionEvaluator(labelCol="next_month_spend", predictionCol="prediction", metricName="mae")
rmse_evaluator = RegressionEvaluator(labelCol="next_month_spend", predictionCol="prediction", metricName="rmse")
r2_evaluator = RegressionEvaluator(labelCol="next_month_spend", predictionCol="prediction", metricName="r2")

mae = mae_evaluator.evaluate(regression_predictions)
rmse = rmse_evaluator.evaluate(regression_predictions)
r2 = r2_evaluator.evaluate(regression_predictions)

print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²  :", round(r2, 4))

+-----------+----------------+------------------+
|customer_id|next_month_spend|        prediction|
+-----------+----------------+------------------+
|     100003|          3677.2|3432.5004708742367|
|     100007|         5304.81| 5254.429445251664|
|     100009|         5351.79| 5146.544695430954|
|     100014|         3595.57|3608.9534663744944|
|     100020|         4790.87| 4771.283214235013|
|     100024|         6359.19| 6216.015811610577|
|     100030|         5633.16| 5602.086355256611|
|     100036|         5735.11|   5223.6423924336|
|     100046|         4510.57| 4307.634136246681|
|     100047|         2838.79|3264.3476662878043|
+-----------+----------------+------------------+
only showing top 10 rows

MAE : 202.02
RMSE: 251.67
R²  : 0.9428


In [24]:
reg_output = regression_predictions.select(
    "customer_id", "age", "income", "tenure_months", "monthly_spend",
    "usage_hours", "support_calls", "satisfaction", "churn",
    "next_month_spend", "prediction"
)
reg_pdf = reg_output.toPandas()
reg_pdf = reg_pdf.rename(columns={"prediction": "predicted_next_month_spend"})

reg_file = os.path.join(Assignment_DIR, "06_regression_predictions.csv")
reg_pdf.to_csv(reg_file, index=False)
print("Saved:", reg_file)

regression_metrics = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R2"],
    "Value": [mae, rmse, r2],
    "Model": ["Linear Regression"] * 3
})

reg_metrics_file = os.path.join(Assignment_DIR, "07_regression_metrics.csv")
regression_metrics.to_csv(reg_metrics_file, index=False)
regression_metrics

Saved: /content/AI_ML_PowerBI_Assignment/06_regression_predictions.csv


,Metric,Value,Model
0,MAE,202.019343,Linear Regression
1,RMSE,251.672902,Linear Regression
2,R2,0.942836,Linear Regression


In [25]:
model_summary = pd.DataFrame({
    "Task": ["Classification", "Regression"],
    "Model": ["Logistic Regression", "Linear Regression"],
    "Primary_Metric": [f1, r2],
    "Execution_Status": ["Completed", "Completed"]
})

summary_file = os.path.join(Assignment_DIR, "08_model_summary.csv")
model_summary.to_csv(summary_file, index=False)
model_summary

,Task,Model,Primary_Metric,Execution_Status
0,Classification,Logistic Regression,0.978967,Completed
1,Regression,Linear Regression,0.942836,Completed


In [26]:
overall_metrics = pd.DataFrame([
    {"Task": "Classification", "Model": "Logistic Regression", "Metric": "Accuracy", "Value": accuracy},
    {"Task": "Classification", "Model": "Logistic Regression", "Metric": "Precision", "Value": precision},
    {"Task": "Classification", "Model": "Logistic Regression", "Metric": "Recall", "Value": recall},
    {"Task": "Classification", "Model": "Logistic Regression", "Metric": "F1 Score", "Value": f1},
    {"Task": "Classification", "Model": "Logistic Regression", "Metric": "ROC-AUC", "Value": auc},
    {"Task": "Regression", "Model": "Linear Regression", "Metric": "MAE", "Value": mae},
    {"Task": "Regression", "Model": "Linear Regression", "Metric": "RMSE", "Value": rmse},
    {"Task": "Regression", "Model": "Linear Regression", "Metric": "R2", "Value": r2},
])

overall_file = os.path.join(Assignment_DIR, "09_overall_model_metrics.csv")
overall_metrics.to_csv(overall_file, index=False)
overall_metrics

,Task,Model,Metric,Value
0,Classification,Logistic Regression,Accuracy,0.985945
1,Classification,Logistic Regression,Precision,0.972087
2,Classification,Logistic Regression,Recall,0.985945
3,Classification,Logistic Regression,F1 Score,0.978967
4,Classification,Logistic Regression,ROC-AUC,0.752708
5,Regression,Linear Regression,MAE,202.019343
6,Regression,Linear Regression,RMSE,251.672902
7,Regression,Linear Regression,R2,0.942836


In [27]:
print("Generated files:\n")
for file in sorted(os.listdir(Assignment_DIR)):
    path = os.path.join(Assignment_DIR, file)
    size = os.path.getsize(path) / 1024
    print(f"{file:45s} {size:8.2f} KB")

Generated files:

02_clean_customer_data.csv                      513.51 KB
03_classification_predictions.csv               125.13 KB
04_classification_metrics.csv                     0.25 KB
05_confusion_matrix.csv                           0.06 KB
06_regression_predictions.csv                   132.95 KB
07_regression_metrics.csv                         0.14 KB
08_model_summary.csv                              0.16 KB
09_overall_model_metrics.csv                      0.48 KB
classification_model                              4.00 KB
regression_model                                  4.00 KB


In [28]:
import shutil

zip_path = shutil.make_archive("/content/AI_ML_PowerBI_Assignment", "zip", Assignment_DIR)
print("ZIP created:", zip_path)

ZIP created: /content/AI_ML_PowerBI_Assignment.zip
